In [1]:
from IPython.display import Image
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends import backend_pdf as bpdf
import itertools
import numpy as np
import pandas as pd

import pickle as pkl
import sys
import os

# Get the current working directory (where the script is run from)
current_dir = os.getcwd()

# Get the parent directory
parent_dir = os.path.dirname(current_dir)

# Add the parent directory to sys.path
sys.path.append(parent_dir)

params = {'legend.fontsize': 15,
          'figure.figsize': (7, 7),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'axes.linewidth': 3,
          'xtick.labelsize':18,
          'ytick.labelsize':18,
          'xtick.labelsize':18,
          'ytick.labelsize':18,
          'svg.fonttype':'none'}
plt.rcParams.update(params)

In [2]:
def process_communities(path_csv, test_size=1000, subset_size=1000, k=5, seed=1234):
    np.random.seed(seed)
    df = pd.read_csv(path_csv)

    species = df.columns[df.columns.str.contains('S')].to_list()
    treatments = df.Treatments.unique().copy()

    include_train_always = treatments[:(len(species) * 2) + 1].copy()
    treatments = treatments[(len(species) * 2) + 1:].copy()
    np.random.shuffle(treatments)

    test_comm = treatments[:test_size]
    train_comm = treatments[test_size:]
    ars = train_comm[:k*(len(train_comm)//k)].reshape(k, -1)
    train_comms = np.empty((k, subset_size), dtype=object)
    for i in range(k):
        train_comms[i, :(len(species) * 2) + 1] = include_train_always
        train_comms[i, (len(species) * 2) + 1:] = ars[i,:subset_size - (len(species) * 2) - 1]

    return test_comm, train_comms

In [3]:
species = [12, 30, 50]
complexity = ['intricate', 'simple', 'base']
case_ = ['baseline', 'sparse', 'noisy']
experiments = [100, 200, 300, 500, 1000]

for s in species:
    if s == 12:
        exps = 4000
    else:
        exps = 6000
    for cx in complexity:
        for cs in case_:
            path_csv = f'{parent_dir}/coalescence/community-simulator/syn_coms/data_{s}s15r_{exps}exps_{cx}_{cs}.csv'
            if s == 12:
                test_comm, train_comms = process_communities(path_csv, subset_size=500)
            else:
                test_comm, train_comms = process_communities(path_csv)
            df = pd.read_csv(path_csv)

            test_subset = df[df['Treatments'].isin(test_comm)]
            test_subset.to_csv(f'datasets/data_{s}s15r_{cx}_{cs}_test.csv', index=False)
            for exp in experiments:
                if exp == 1000 and s == 12:
                    continue
                for k in range(len(train_comms)):
                    train_subset = df[df['Treatments'].isin(train_comms[k,:exp])]
                    train_subset.to_csv(f'datasets/data_{s}s15r_{cx}_{cs}_{exp}exps_{k+1}.csv', index=False)


In [4]:
path_csv = f'{parent_dir}/coalescence/community-simulator/syn_coms/data_12s15r_4000exps_intricate_baseline.csv'
test_size=1000
subset_size=1000
k=3
seed=1234

for cx in complexity:
        for cs in case_:
            np.random.seed(seed)
            path_csv = f'{parent_dir}/coalescence/community-simulator/syn_coms/data_12s15r_4000exps_{cx}_{cs}.csv'
             
            df = pd.read_csv(path_csv)

            species = df.columns[df.columns.str.contains('S')].to_list()
            treatments = df.Treatments.unique().copy()

            include_train_always = treatments[:(len(species) * 2) + 1].copy()
            treatments = treatments[(len(species) * 2) + 1:].copy()
            np.random.shuffle(treatments)

            test_comm = treatments[:test_size]
            train_comm = treatments[test_size:]
            ars = train_comm[:k*(len(train_comm)//k)].reshape(k, -1)
            train_comms = np.empty((k, subset_size), dtype=object)
            for i in range(k):
                train_comms[i, :(len(species) * 2) + 1] = include_train_always
                train_comms[i, (len(species) * 2) + 1:] = ars[i,:subset_size - (len(species) * 2) - 1]
                train_subset = df[df['Treatments'].isin(train_comms[i,:exp])]
                train_subset.to_csv(f'datasets/data_12s15r_{cx}_{cs}_{subset_size}exps_{i+1}.csv', index=False)


In [5]:
# Get the current working directory (where the script is run from)
current_dir = os.getcwd()

# Get the parent directory
parent_dir = os.path.dirname(current_dir)

# Add the parent directory to sys.path
sys.path.append(parent_dir)
import re
import glob

np.random.seed(1234)
species_n = [12, 30, 50]
complexity = ['intricate', 'simple', 'base']
case_ = ['baseline', 'sparse', 'noisy']
experiments = [100, 200, 300, 500, 1000]
mediators = {'mediators1': ['R01'], 
             'mediators2': ['R09'], 
             'mediators3': ['R01', 'R05'], 
             'mediators4': ['R01', 'R13'], 
             'mediators5': ['R01', 'R03', 'R07', 'R10', 'R13'], 
             'mediators6': ['R01', 'R03', 'R05', 'R07', 'R08', 'R09', 'R10', 'R12', 'R13', 'R15'], 
             'mediators7': ['R01', 'R02', 'R03', 'R04', 'R05', 'R06', 'R07', 'R08', 'R09', 'R10', 'R11', 'R12', 'R13', 'R14', 'R15']}

for s in species_n:
    r = 15
    lookup = {
            'memory': [],
            'path_csv': [],
            'mediator_name': [],
            'mediators': [],
            }
    for cx in complexity:
        for cs in case_:
            for name, mediator in mediators.items():
                # if name in ['mediators6', 'mediators7'] and s != 12:
                #     continue
                for exp in experiments:
                    if exp == 1000 and s == 12:
                        ks = 3
                    else:
                        ks = 5
                    for k in range(ks):
                        path_csv = f'datasets/data_{s}s15r_{cx}_{cs}_{exp}exps_{k+1}.csv'
                        if cs == 'sparse' and exp in [100, 200, 300, 500] and name not in ['mediators6', 'mediators7']:
                            mem = 'small'
                        elif cs == 'sparse' and exp in [100, 200, 300, 500] and name in ['mediators6', 'mediators7']:
                            mem = 'medium'
                        if exp in [100, 200, 300] and name not in ['mediators6', 'mediators7']:
                            mem = 'small'
                        elif (exp == 500 and name not in ['mediators6', 'mediators7']) or (exp in [100, 200, 300] and name in ['mediators6', 'mediators7']):
                            mem = 'medium'
                        elif exp == 1000 or (exp == 500 and name in ['mediators6', 'mediators7']):
                            mem = 'large'
                        lookup['memory'].append(mem)
                        lookup['path_csv'].append(path_csv)
                        lookup['mediator_name'].append(name)
                        mediator_str = ', '.join(mediator)
                        mediator_str = mediator_str.replace(' ', '')
                        lookup['mediators'].append(mediator_str)
                    lookup_df = pd.DataFrame(lookup)
                    lookup_df['mediators_n'] = lookup_df['mediators'].apply(lambda x: len(re.findall(',', x)) + 1)
                    order = ['small', 'medium', 'large']
                    lookup_df['memory'] = pd.Categorical(lookup_df['memory'], categories=order, ordered=True)
                    lookup_df = lookup_df.sort_values(by=['memory', 'mediators_n'])
                    lookup_df = lookup_df.drop(columns=['mediators_n'])
                    lookup_df.to_csv(f'lookup-files/lookup_{s}s{r}r.txt', header=False, index=False, sep=' ')